# RKLLM 변환 환경 (Hugging Face LLM → RK3588 W8A8)

- 목적: Colab(x86_64 Linux)에서 **rkllm-toolkit 1.3.0**(학교 변환 도커·보드 런타임과 같은 버전)을 설치하고 소형 LLM 1개를 `.rkllm`으로 변환한다. (PLAN BOARD-06)
- 런타임: **GPU(T4) 권장** (CPU도 가능하지만 느림). Python 3.10~3.12만 지원
- **새 런타임에서 실행**: rknn-toolkit2와 의존성이 충돌한다 (torch 2.6.0 vs ≤2.2.0, protobuf ≥4.21 vs 3.20.3)
- 샘플 모델: `Qwen/Qwen3-0.6B` (RK3588 공식 벤치마크에 있는 모델). 우리 후보 비교는 LLM-06에서
- 절차·근거: `recipes/llm-to-rkllm.md`
- 끝나면: 출력의 버전·시간·파일 크기를 이슈 #13에 댓글로 남긴다.
- **기본 경로는 학교 제공 A100 서버의 변환 도커(`run.sh`)다.** 이 노트북은 서버를 쓸 수 없을 때의 대체 경로 (`recipes/llm-to-rkllm.md`)


In [ ]:
import platform
import sys

print(sys.version)  # 3.10 ~ 3.12 이어야 한다
print(platform.machine())  # x86_64 이어야 한다
!nvidia-smi -L || echo "GPU 없음 → 아래 변환 셀에서 --device cpu"

In [ ]:
# JARVIS 저장소 (변환 스크립트)
!git clone -q https://github.com/samsung-team-jarvis/jarvis.git /content/jarvis || (cd /content/jarvis && git pull -q)

In [ ]:
# rkllm-toolkit: 공식 requirements → 휠 순서로 설치
RKLLM_TAG = "release-v1.3.0"  # 보드 런타임(librkllmrt 1.3.0)과 맞춘다
RKLLM_VERSION = "1.3.0"
import sys

tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
base = f"https://raw.githubusercontent.com/airockchip/rknn-llm/{RKLLM_TAG}/rkllm-toolkit/packages"
!pip install -q -r {base}/requirements.txt
!pip install -q {base}/rkllm_toolkit-{RKLLM_VERSION}-{tag}-{tag}-linux_x86_64.whl

설치 후 import 오류가 나면 **런타임 → 세션 다시 시작** 후 아래 셀부터 실행한다 (torch 버전이 바뀌기 때문).

In [ ]:
from rkllm.api import RKLLM

print("rkllm-toolkit import OK")

In [ ]:
# 샘플 모델과 캘리브레이션 데이터 (공식 예제의 data_quant.json)
from huggingface_hub import snapshot_download

MODEL_DIR = snapshot_download("Qwen/Qwen3-0.6B", local_dir="/content/models/Qwen3-0.6B")
!wget -q -O /content/data_quant.json https://raw.githubusercontent.com/airockchip/rknn-llm/release-v1.3.0/examples/rkllm_api_demo/export/data_quant.json
!ls -lh /content/data_quant.json

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
!mkdir -p /content/out
!python /content/jarvis/training/convert/hf_to_rkllm.py \
    --model {MODEL_DIR} --dataset /content/data_quant.json \
    --out /content/out/qwen3-0.6b_w8a8_rk3588.rkllm --device {DEVICE}

## 우리 모델로 바꾸려면

1. LoRA 학습 후 병합한 fp16 모델 디렉터리를 `--model`에 (또는 `--lora`로 분리 변환)
2. 캘리브레이션: **Train 발화**로 만든 `quant.json` (input에는 추론과 같은 chat template 적용)
3. YOLO와 NPU를 나눠 쓸 경우 `--npu-cores` 조정 (architecture §6, BOARD-08)

In [ ]:
# (선택) 결과를 팀 Drive에 저장. 가중치는 git에 올리지 않는다.
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/jarvis/models"
!cp -v /content/out/* "/content/drive/MyDrive/jarvis/models/"